**Multistage Processes**

Multi-effect evaporator for ZLD (Zero Liquid Discharge) Plant.
A facility needs to treat 5000kg/hr of a salt solution, which has 12% salt conc, which has to brought to 25% salt conc, using N effect evaporators.

Boiling point elevation = 5 degC (i.e, at 1 atm, boiling point = 105 deg C)

Inputs:

Feed rate (F) = 5000 kg/hr,
$C_0$ =  0.12 (wt.fr), $C_3 = 0.25$ (wt.fr)

Latent heat of evaporation of steam = 540 kcal/kg

Salt Balance: $F_1C_1 = F_4C_4$

Total Evaporation: $F_1 - F_4$

Heat load in each effect = $Q_{H1} = (F_1 - F_2)*lambda$ and so on

**Assumption: Equal evaporation in all effects**

In [ ]:
!pip install CoolProp
import CoolProp.CoolProp as CP
import numpy as np
from scipy.optimize import fsolve
import pandas as pd

In [ ]:
F = np.array([5000, 0, 0, 0]) # Array of mass flow rates, concentrate streams
C = np.array([0.12, 0, 0, 0.25]) # Array of concentrations, concentrate streams
T_liq = np.array([105, 0, 0, 0]) # Array of temperatures, concentrate streams
P_atm = np.array([1, 0.55, 0.25]) # Column pressure in atm
P_steam = 4 #Pressure of fresh steam
T_steam = 1 #Temperature of fresh steam
U = np.array([1000, 800, 600]) #Initial assumptions for overall heat transfer coefficient in kcal/(m2*hr*degC)
BPE = 5 #Boiling point elevation
lt_heat = 540

In [ ]:
#To calculate salt conc in exit stream of Nth evaporator
def salt(F_inlet, C_inlet, C_outlet):
  F_outlet = F_inlet* C_inlet/C_outlet
  return F_outlet
F[-1] = salt(F[0], C[0], C[-1])
print(f"Mass flow rate of concentrate in exit stream of Nth evaporator is {F[-1]} kg/hr")

Mass flow rate of concentrate in exit stream of Nth evaporator is 2400 kg/hr


In [ ]:
#Calculating mass flow rate, concentration (for concentrate streams) and heat load
total_evap = F[0] - F[-1]
print(f"Total evaporation is {total_evap} kg")
N = 3 #No. of effects
evap_per_effect = total_evap/N #Equal evaporation per effect
print(f"Total evaporation per effect is {evap_per_effect}")

Q = [0, 0, 0]

for i in range(N):
  F[i+1] = F[i] - evap_per_effect
  C[i+1] = (F[i]*C[i])/F[i+1]
  Q[i] = lt_heat*(F[i]-F[i+1])
print(f"Mass flow rates: {F} kg/hr")
print(f"Concentrations: {C}")
print(f"Heat load: {Q} kcal/hr")

Total evaporation is 2600 kg
Total evaporation per effect is 866.6666666666666
Mass flow rates: [5000 4133 3266 2399] kg/hr
Concentrations: [0.12       0.145173   0.18371096 0.25010421]
Heat load: [np.int64(468180), np.int64(468180), np.int64(468180)] kcal/hr


In [ ]:
#Using COOLPROP to calculate temperatures
P_SI = [101325*1, 101325*0.55, 101325*0.25] #SI units

T = np.array([CP.PropsSI("T", "P", p, "Q", 0, "Water") for p in P_SI])

print(f"Temperatures (K) : {T}")
T_L = np.zeros(N)
T_V = np.zeros(N)
T_L = T + BPE
T_V = T
print(f"Temperatures of liquid(K): {T_L} ")
print(f"Temperatures of vapour(K): {T_V} ")

Temperatures (K) : [373.12429585 357.19262633 338.40734609]
Temperatures of liquid(K): [378.12429585 362.19262633 343.40734609] 
Temperatures of vapour(K): [373.12429585 357.19262633 338.40734609] 


In [ ]:
#Calculate temperature of steam, driving force (temperature diff)
del_T = np.zeros(N)
P_u = 4*101325
T_u = CP.PropsSI("T", "P", P_u, "Q", 1, "Water")
for i in range(N):
  if i == 0:
    del_T[i] = T_u - T_L[i]
  else:
    del_T[i] = T_V[i-1] - T_L[i]
print(f"Temperature difference between liquid and vapour streams: {del_T}")

Temperature difference between liquid and vapour streams: [39.10914573 10.93166951 13.78528025]


In [ ]:
#Area of heat transfer
A = np.zeros(N)
for i in range(N):
  A[i] = Q[i]/(U[i]*del_T[i])
print(f"Area of heat transfer in m2: {A}") #Area

Area of heat transfer in m2: [11.97111293 53.53482369 56.6038547 ]


In [ ]:
data = {
    "Effect Number" : [1, 2, 3],
    "Liquid in (kg/hr)" : [F[0], F[1], F[2]],
    "Liquid out (kg/hr)" : [F[1], F[2], F[3]],
    "Vapour flow (kg/hr)" : evap_per_effect,
    "Liquid temperature(K)": [T_L[0], T_L[1], T_L[2]],
    "Vapour temperature(K)" : [T_V[0], T_V[1], T_V[2]],
    "Pressure (atm)" : [P_atm[0], P_atm[1], P_atm[2]],
    "Overall Heat Transfer Coefficient": [U[0], U[1], U[2]],
    "Area of heat transfer(m2)": [A[0], A[1], A[2]]

}
df = pd.DataFrame(data)
print(df)

   Effect Number  Liquid in (kg/hr)  Liquid out (kg/hr)  Vapour flow (kg/hr)  \
0              1               5000                4133           866.666667   
1              2               4133                3266           866.666667   
2              3               3266                2399           866.666667   

   Liquid temperature(K)  Vapour temperature(K)  Pressure (atm)  \
0             378.124296             373.124296            1.00   
1             362.192626             357.192626            0.55   
2             343.407346             338.407346            0.25   

   Overall Heat Transfer Coefficient  Area of heat transfer(m2)  
0                               1000                  11.971113  
1                                800                  53.534824  
2                                600                  56.603855  


Heat Transfer Calculations

In [ ]:
# Dimensions in mm
tube_id = 19
tube_t = 1.5
tube_len = 6000
tube_od = tube_id + (2*tube_t)
HTArea_per_tube = np.pi*tube_od*tube_len ##mm2
CSArea_per_tube = np.pi*(tube_id**2)/4 #mm2
N_tubes = A/(HTArea_per_tube*1e-06)
vel_tube = 1 #m/s
recirculation_flow = N_tubes*vel_tube*(CSArea_per_tube*1e-06)
print(recirculation_flow)
print(f"Number of tubes required: {N_tubes}")

[0.0081848  0.03660241 0.03870074]
Number of tubes required: [ 28.86760298 129.09593663 136.49671625]
